## 🏗️ LeetCode 253: Meeting Rooms II
---

<div style="padding: 15px; border-left: 8px solid #9C27B0; background-color: #f3e5f5; color: #4a148c; border-radius: 4px;">
    <strong>The Core Insight:</strong> We need the peak number of simultaneously running meetings. Use a min-heap of end times. For each new meeting (sorted by start), check if the earliest-ending room is free. If yes, reuse it (heapreplace). If no, open a new room (heappush). The heap size = number of rooms in use.
</div>

### 🛠️ The Mathematical Model

The minimum rooms needed = the maximum simultaneous meetings at any point in time.
We track "rooms in use" via a min-heap of end times — the root is the room that becomes free soonest.

$$\text{min rooms} = \max_{t} |\{[s_i, e_i] : s_i \leq t < e_i\}|$$

---

### 📋 Problem

Given an array of meeting time intervals `[start, end]`, find the minimum number of conference rooms required.

**Example 1:**
```
Input:  [[0,30],[5,10],[15,20]]
Output: 2
```

**Example 2:**
```
Input:  [[7,10],[2,4]]
Output: 1
```

**Constraints:** 1 ≤ intervals.length ≤ 10⁴ | 0 ≤ start_i < end_i ≤ 10⁶

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Room Reuse</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"As each meeting starts, check if any room is free. The room that finishes soonest is checked first. If it's done, reuse it. If not, open a new room. A min-heap of end times tells us which room finishes soonest."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Min-heap of end times. heap[0] = earliest-ending room. If heap[0] ≤ new_start: reuse. Else: new room.</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Peak Concurrency</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"The heap size at any point = number of rooms currently in use. The maximum heap size over all time = the minimum rooms needed."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">heappush adds a room. heapreplace reuses a room (no size change). max(len(heap)) = answer.</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> Checking all time points for concurrent meetings is O(T × n) where T is the time range. Min-heap approach: O(n log n) — sort once, process each meeting once.
</div>

## 🐢 Approach 1: Brute Force — $O(T \times n)$

In [ ]:
def minMeetingRooms_brute(intervals):
    """
    Brute Force: Check each time point for concurrent meetings
    Time: O(T * n) where T = max time range | Space: O(T)
    """
    if not intervals:
        return 0
    max_time = max(e for _, e in intervals)
    max_rooms = 0
    for t in range(max_time + 1):
        concurrent = sum(1 for s, e in intervals if s <= t < e)
        max_rooms = max(max_rooms, concurrent)
    return max_rooms


print(minMeetingRooms_brute([[0,30],[5,10],[15,20]]))   # Expected: 2
print(minMeetingRooms_brute([[7,10],[2,4]]))             # Expected: 1

## 🔬 Complexity Analysis: $O(T \times n)$ vs. $O(n \log n)$

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> We only care about "room availability" at meeting START times — not every point in time. Sort meetings by start. At each start, check if the earliest-ending room (min-heap root) is available. If yes: reuse it. If no: add a room. The heap size tracks current room count.
</div>

---

## 📉 Why Brute Force Fails: The $O(T \times n)$ Trap

Checking every time unit up to max_time (potentially 10⁶) × n meetings = up to 10¹⁰ operations.

| Input Type | Brute Force Performance | Reason |
|------------|------------------------|--------|
| **max_time = 10⁶, n = 10⁴** | 10¹⁰ checks | One check per time unit per meeting |
| **Long meetings** | Slow | Each long meeting counted at every time unit |

---

## 🚀 The Optimal Approach: $O(n \log n)$

Sort by start time. Maintain a min-heap of end times (each occupied room's end time). For each meeting:
- If `heap[0] <= start`: earliest room just freed → reuse it (`heapreplace(heap, end)`)
- Else: no room available → open new room (`heappush(heap, end)`)
Final answer: `len(heap)` = rooms in use at peak.

### The Key Lifecycle Rule
1. **Sort meetings by start time** — process in chronological order
2. **For each meeting:** check if the soonest-finishing room is free
3. **Reuse or open** — heapreplace (reuse) or heappush (new room)

---

## ✅ Mathematical Proof

The min-heap always contains exactly the set of currently occupied rooms' end times. When we process meeting i (start_i), heap[0] is the earliest that any room becomes free. If heap[0] <= start_i, that room is free — we can reuse it. If not, all rooms are still occupied — we must add one. At the end, len(heap) = number of rooms needed for the peak concurrent period.

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> The heap encodes the "earliest available room." Reuse when available, add when not. len(heap) at end = minimum rooms needed.
</div>

## 🚀 Approach 2: Min-Heap of End Times — $O(n \log n)$
---

Instead of checking every time point, we process meetings in **start-time order** and track **room end times** in a min-heap.

As we iterate:
1. Sort by start time
2. For each meeting: check heap[0] (earliest room to free)
3. If heap[0] ≤ start: heapreplace (reuse room). Else: heappush (new room)
4. len(heap) = final room count

In [ ]:
import heapq

def minMeetingRooms(intervals: list[list[int]]) -> int:
    """
    Optimal: Min-Heap of end times
    Time: O(n log n) | Space: O(n) for heap
    """
    if not intervals:
        return 0
    intervals.sort(key=lambda x: x[0])   # sort by start time
    heap = []   # min-heap of end times (each entry = end time of an occupied room)

    for start, end in intervals:
        if heap and heap[0] <= start:    # earliest room is free
            heapq.heapreplace(heap, end) # reuse that room (pop old end, push new end)
        else:
            heapq.heappush(heap, end)    # all rooms occupied — open new room

    return len(heap)


print("Optimal:", minMeetingRooms([[0,30],[5,10],[15,20]]))   # Expected: 2
print("Optimal:", minMeetingRooms([[7,10],[2,4]]))             # Expected: 1
print("Optimal:", minMeetingRooms([[1,5],[5,10]]))             # Expected: 1 (back-to-back, not concurrent)

## 🎤 5 Interview Q&A

### Q1: How does this relate to capacity planning?

**Answer:** Replace "meetings" with "ETL jobs," "monitoring agents," or "database queries." The algorithm tells you the minimum number of worker threads or servers needed to handle peak concurrent workload. At Citi: replace meetings with APM polling intervals — what's the minimum number of polling agents needed to monitor 6,000 servers with no gaps?

---

### Q2: What does `heapreplace` do vs pop + push?

**Answer:** `heapreplace(heap, val)` is an atomic pop + push in O(log n) — it pops the minimum and pushes the new value in a single heap operation. This is more efficient than `heappop` + `heappush` (which would be two O(log n) operations). Use `heapreplace` when you know you'll always push after popping (the room is always reused, not discarded).

---

### Q3: What is the time complexity of the optimal approach and why?

**Answer:** O(n log n). Sorting: O(n log n). Processing: n meetings, each does one heap operation (heapreplace or heappush) in O(log n). Total: O(n log n + n log n) = O(n log n). The heap never grows beyond n elements.

---

### Q4: Why sort by start time and not end time?

**Answer:** We process meetings as they start — chronologically. At each meeting's start time, we need to know if any room has become free (end time ≤ current start time). Sorting by start time means we process meetings in the order they begin, which is the natural order for room allocation decisions.

---

### Q5: What are the edge cases to watch for?

**Answer:** (1) Empty intervals — early return 0. (2) Back-to-back meetings (end = next start) — `heap[0] <= start` uses `<=`, so [0,5] and [5,10] correctly reuse the same room. (3) All at same time — heap grows to n, returns n. (4) All sequential — heap never grows past 1, returns 1.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **Peak Concurrency** | The maximum number of meetings/tasks/jobs running simultaneously — what Meeting Rooms II computes |
| **Min-Heap of End Times** | Tracks occupied rooms by their end times — root = earliest room to become free |
| **heapreplace** | Atomic heappop + heappush — removes minimum, inserts new value, single O(log n) operation |
| **Room Reuse** | When heap[0] ≤ new start: the earliest-finishing room is free and can be reassigned |
| **Back-to-back** | Meetings where one ends exactly when the next begins — these are typically non-concurrent (use ≤ for reuse) |
| **Worker Pool** | The heap size = number of workers/rooms/threads needed at peak — the answer to capacity questions |
| **Sort by Start** | Process meetings in chronological order — necessary for correct room allocation decisions |
| **Concurrent** | Multiple meetings occupying different rooms simultaneously — the heap encodes this state |

## 💼 The Citi Narrative

**Context:** Citi's monitoring infrastructure needed to determine how many APM polling agents were needed to cover 6,000 servers — each server had a polling window [start, end) during which it expected a heartbeat check.

**Scenario:** 6,000 polling windows with various start/end times. At peak morning hours, many windows overlapped. The minimum number of polling agents = peak concurrent polling windows = Meeting Rooms II on polling intervals.

**How this pattern applied:** Sorted 6,000 polling intervals by start time. Min-heap of end times tracked active polling windows. At peak morning, heap grew to 420 — meaning 420 concurrent polling windows. This set the minimum agent pool size.

**Impact:** Instead of provisioning 6,000 agents (one per server), the algorithm showed 420 agents sufficed for full coverage — a 93% reduction in infrastructure cost. Agent pool was right-sized to peak demand, with 10% buffer added: 462 agents deployed instead of 6,000.

In [ ]:
# -------------------------------------------------------
# PRACTICE: Given CPU job intervals [start, end],
# find the MINIMUM number of CPU cores needed to run
# all jobs (no job can be split across cores).
# Same algorithm as Meeting Rooms II.
# -------------------------------------------------------

import heapq

def minCpuCores(jobs):
    # Your solution here — same as minMeetingRooms
    pass


# Test
print(minCpuCores([[0,3],[1,4],[2,5]]))    # Expected: 3 (all concurrent at t=2)
print(minCpuCores([[0,2],[2,4],[4,6]]))    # Expected: 1 (sequential)

## 🎯 Summary: Key Takeaways

### The Pattern
**Min-Heap of End Times** — Track occupied rooms; reuse the earliest-finishing room; heap size = current room count

### When to Use It
- ✅ Minimum resources needed for peak concurrent workload
- ✅ Meeting rooms, CPU cores, server agents, worker threads
- ✅ Any "how many do I need simultaneously?" problem
- ❌ **Don't use when:** You want to maximize throughput differently — use LC #435 (greedy by end)

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force (scan all times) | $O(T \times n)$ | $O(T)$ |
| Optimal (Min-Heap) | $O(n \log n)$ | $O(n)$ |

### Interview Confidence Checklist
- [ ] Can explain the min-heap of end times approach
- [ ] Can explain heapreplace and why it's used
- [ ] Can write the solution from memory in 4 minutes
- [ ] Can explain the relationship to capacity planning
- [ ] Can state the back-to-back edge case and how <= handles it

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master Meeting Rooms II and you've solved the algorithmic foundation of capacity planning — minimum resources for peak concurrent demand. 🚀